# 0. 환경 점검 + 경로 정의

In [1]:
from pathlib import Path
import torch
import onnx
import onnxruntime as ort

print(f'PyTorch: {torch.__version__}')
print(f'ONNX:    {onnx.__version__}')
print(f'ORT:     {ort.__version__}')

PROJECT = Path(r'D:\02.study\part4_wj\Battery\Battery_Project')
PT_PATH = PROJECT / 'models' / 'best_finetuned.pt'
SRC_DIR = PROJECT / 'pretrained_aihub' / '1.모델소스코드' / '모델1_DeepLabv3' / 'pytorch-deeplab-xception-eval'
IMG_DIR = PROJECT / 'battery_image'
ONNX_OUT = PROJECT / 'models' / 'battery_deeplab_v1.onnx'

for p in [PT_PATH, SRC_DIR, IMG_DIR]:
    assert p.exists(), f'경로 없음: {p}'
print('경로 확인 완료')

PyTorch: 2.11.0+cpu
ONNX:    1.21.0
ORT:     1.26.0
경로 확인 완료


## 1. DeepLab 모델 재구성 + 가중치 로드

@04_finetune_colab.ipynb와 동일한 구조로 모델을 만든 뒤 fine-tune 가중치를 로드합니다.

**헤드는 3채널로 교체**되어 있어야 state_dict가 일치합니다.

### 1-1. sys.path + monkey-patch + 모델 로드

In [2]:
import sys
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# DRN 백본의 ImageNet pretrained 다운로드 차단 (03/04와 동일)
from modeling.backbone import drn
_orig = drn.drn_d_54
drn.drn_d_54 = lambda BatchNorm, pretrained=False: _orig(BatchNorm, pretrained=False)

from modeling.deeplab import DeepLab
import torch.nn as nn

NUM_CLASSES = 3

# 1) 모델 인스턴스: 일단 4클래스로 생성 후 헤드 교체
model = DeepLab(num_classes=4, backbone='drn', output_stride=16,
                sync_bn=False, freeze_bn=True)
old = model.decoder.last_conv[8]
model.decoder.last_conv[8] = nn.Conv2d(old.in_channels, NUM_CLASSES,
                                       kernel_size=1, stride=1, padding=0, bias=True)

# 2) Fine-tune 가중치 로드
ckpt = torch.load(str(PT_PATH), map_location='cpu', weights_only=False)
result = model.load_state_dict(ckpt['state_dict'])
print(f'학습 epoch: {ckpt["epoch"]},  val mIoU: {ckpt["val_miou"]:.4f}')
print(f'가중치 로드 — Missing: {len(result.missing_keys)}, Unexpected: {len(result.unexpected_keys)}')

model.eval()
print('\nFine-tune 모델 준비 완료 (eval 모드)')

학습 epoch: 28,  val mIoU: 0.4038
가중치 로드 — Missing: 0, Unexpected: 0

Fine-tune 모델 준비 완료 (eval 모드)


## 2. ONNX export

`torch.onnx.export`로 정적 그래프 변환합니다.

입력 shape 고정, opset 18, dynamic_axes 은 None 으로 합니다.

In [3]:
dummy_input = torch.randn(1, 3, 513, 513)

torch.onnx.export(
    model,
    dummy_input,
    str(ONNX_OUT),
    opset_version=18,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes=None,                    # 정적 shape
    do_constant_folding=True,
    export_params=True,
)

print(f'ONNX 저장: {ONNX_OUT}')
print(f'파일 크기: {ONNX_OUT.stat().st_size / 1024**2:.1f} MB')

[torch.onnx] Obtain model graph for `DeepLab([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DeepLab([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


D:\jupyter\Anaconda\install\Lib\copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX 저장: D:\02.study\part4_wj\Battery\Battery_Project\models\battery_deeplab_v1.onnx
파일 크기: 0.3 MB


### 2-2. op별 호환성 점검

In [4]:
try:
    torch.onnx.export(model, dummy_input, str(ONNX_OUT),opset_version=17, verbose=True)
except Exception as e:
    print(f'에러: {e}')

W0513 11:19:43.476000 20172 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `DeepLab([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DeepLab([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


D:\jupyter\Anaconda\install\Lib\copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "d:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "d:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


모델 크기가 작아서 정상적으로 변환이 안된것 같아 확인해보니 정상적으로 변환된 것으로 보입니다.

파일 크기가 0.3MB으로 나오는 것과 `.onnx.data` 파일이 생성된 이유는 사용 하는 PyTorch 2.1+ 버전의 새로운 ONNX 익스포터(Dynamo 기반) 과 가중치 (Weights)를 그래프 구조와 분리하여 저장했기 때문으로 판단됩니다.

현상 분석
* 파일 분리: battery_deeplab_v1.onnx (0.3MB)에는 모델의 '뼈대(연산 그래프)'만 들어있고,
battery_deeplab_v1.onnx.data (약 155MB)에 실제 '근육(가중치)'이 들어 있습니다.

* 용량 확인: .pt 파일이 약 160MB였으므로, 두 파일의 합(0.3 + 155)이 원본과 유사하다는 점은 데이터가 유실되지 않았음을 시사합니다.

* Opset 17 에러: 최신 익스포터는 내부적으로 Opset 18 기능을 사용하려 하며, 이를 억지로 17로 낮추는 과정에서 호환성 문제가 발생한 것입니다.
하지만 로그 마지막에 Optimize the ONNX graph... ✅가 떴으므로 Opset 18 상태로 최종 저장은 완료되었습니다.

In [4]:
import onnxruntime as ort
import numpy as np

# 세션 생성 (이때 .onnx.data 파일이 같은 폴더에 있어야 합니다)
try:
    session = ort.InferenceSession(str(ONNX_OUT), providers=['CPUExecutionProvider'])
    print("ONNX 로드 성공: 가중치와 그래프가 정상적으로 연결되었습니다.")
    
    # 입력 정보 확인
    in_name = session.get_inputs()[0].name
    in_shape = session.get_inputs()[0].shape
    print(f"입력 이름: {in_name}, Shape: {in_shape}")
    # 더미 데이터로 테스트 추론
    dummy_data = np.random.randn(*in_shape).astype(np.float32)
    outputs = session.run(None, {in_name: dummy_data})
    print(f"추론 테스트 성공: 출력 Shape {outputs[0].shape}")

except Exception as e:
    print(f"검증 실패: {e}")

ONNX 로드 성공: 가중치와 그래프가 정상적으로 연결되었습니다.
입력 이름: input, Shape: [1, 3, 513, 513]
추론 테스트 성공: 출력 Shape (1, 3, 513, 513)


## 3. ONNX 모델 구조 검증

`onnx.checker`로 그래프 무결성 확인 + 입출력 메타데이터 출력합니다.

### 3-1. 구조 검증

`onnx.checker`로 그래프 무결성 확인 + 입출력 메타데이터 출력합니다.

In [5]:
import sys, onnxruntime as ort
print('Python   :', sys.executable)
print('ORT path :', ort.__file__)
print('Providers:', ort.get_available_providers())

Python   : d:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Scripts\python.exe
ORT path : d:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\onnxruntime\__init__.py
Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [6]:
onnx_model = onnx.load(str(ONNX_OUT))
onnx.checker.check_model(onnx_model)
print('ONNX 구조 검증 통과')

print(f'\nIR version : {onnx_model.ir_version}')
print(f'Opset      : {onnx_model.opset_import[0].version}')
print(f'Producer   : {onnx_model.producer_name} {onnx_model.producer_version}')

print('\n[입력]')
for inp in onnx_model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f'  {inp.name}  shape={shape}  dtype={inp.type.tensor_type.elem_type}')

print('\n[출력]')
for out in onnx_model.graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f'  {out.name}  shape={shape}  dtype={out.type.tensor_type.elem_type}')

# op 종류 통계
from collections import Counter
op_count = Counter(node.op_type for node in onnx_model.graph.node)
print(f'\n총 노드: {sum(op_count.values())}개')
print(f'주요 op (상위 10):')
for op, cnt in op_count.most_common(10):
    print(f'  {op:20s} {cnt}')

ONNX 구조 검증 통과

IR version : 10
Opset      : 18
Producer   : pytorch 2.11.0+cpu

[입력]
  input  shape=[1, 3, 513, 513]  dtype=1

[출력]
  logits  shape=[1, 3, 513, 513]  dtype=1

총 노드: 151개
주요 op (상위 10):
  Conv                 67
  Relu                 62
  Add                  16
  Resize               3
  Concat               2
  ReduceMean           1


## 4. PyTorch vs ONNX 수치 비교 (정합성 검증)

같은 입력에 대해 두 추론 엔진의 출력이 **부동소수 오차 범위 내**에서 일치하는지 확인합니다.

통상 `1e-3` 이하 max abs diff면 합격으로 판단합니다.

### 4-1. 출력 이름 가져오기

In [7]:
import onnxruntime as ort
import numpy as np
import torch

# 1. 세션 생성
sess = ort.InferenceSession(str(ONNX_OUT), providers=['CPUExecutionProvider'])

# 2. 모델의 실제 입력/출력 이름 확인
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name
print(f"ONNX 모델 실제 입력 이름: {input_name}")
print(f"ONNX 모델 실제 출력 이름: {output_name}")

ONNX 모델 실제 입력 이름: input
ONNX 모델 실제 출력 이름: logits


### 4-2. dummy 입력 비교

In [8]:
sess = ort.InferenceSession(str(ONNX_OUT), providers=['CPUExecutionProvider'])

torch.manual_seed(0)
x = torch.randn(1, 3, 513, 513)

with torch.no_grad():
    pt_out = model(x).numpy()
onnx_out = sess.run([output_name], {'input': x.numpy()})[0]

import numpy as np
diff = np.abs(pt_out - onnx_out)
print(f'PyTorch out shape: {pt_out.shape}')
print(f'ONNX    out shape: {onnx_out.shape}')
print(f'Max abs diff : {diff.max():.6e}')
print(f'Mean abs diff: {diff.mean():.6e}')

assert diff.max() < 1e-3, '수치 차이가 너무 큼! 변환 설정 재검토 필요'
print('\n수치 정합성 검증 통과 (max diff < 1e-3)')

PyTorch out shape: (1, 3, 513, 513)
ONNX    out shape: (1, 3, 513, 513)
Max abs diff : 1.347065e-05
Mean abs diff: 1.386195e-06

수치 정합성 검증 통과 (max diff < 1e-3)


### 4-3. 실제 결함 이미지로 검증

In [9]:
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF

normalize = T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

# 결함 있는 샘플 3장으로 비교
test_imgs = list(IMG_DIR.glob('RGB_cell_cylindrical_*.png'))[:3]

for img_path in test_imgs:
    img = Image.open(img_path).convert('RGB')
    x = normalize(TF.to_tensor(TF.resize(img, [513, 513]))).unsqueeze(0)

    with torch.no_grad():
        pt = model(x).numpy()
    on = sess.run([output_name], {'input': x.numpy()})[0]

    pt_argmax = pt.argmax(axis=1)
    on_argmax = on.argmax(axis=1)
    agree = (pt_argmax == on_argmax).mean()

    print(f'{img_path.stem:40s}  max_diff={np.abs(pt-on).max():.2e}  argmax 일치={agree*100:.2f}%')

RGB_cell_cylindrical_0722_206             max_diff=1.37e-04  argmax 일치=100.00%
RGB_cell_cylindrical_0729_039             max_diff=2.14e-04  argmax 일치=100.00%
RGB_cell_cylindrical_0729_189             max_diff=1.96e-04  argmax 일치=100.00%


> **기대**: argmax 일치 99.9% 이상.<br/> **실제** : argmax 100% 일치

## 5. 추론 시간 측정

### 5-1. ORT CPU 벤치마크(CPU)

In [14]:
import time

# warm-up 5회
for _ in range(5):
    sess.run([output_name], {'input': x.numpy()})

# 측정 20회
N = 20
t0 = time.perf_counter()
for _ in range(N):
    sess.run([output_name], {'input': x.numpy()})
elapsed = time.perf_counter() - t0

print(f'ORT CPU 추론: {elapsed/N*1000:.1f} ms/image ({N/elapsed:.1f} FPS)')

ORT CPU 추론: 532.2 ms/image (1.9 FPS)


> ONNX Runtime CPU 환경에서 평균 532ms/image(1.9 FPS)의 추론 속도를 확인하였습니다. <br/> 이는 제조 검사 환경의 실시간 처리 요구사항(100ms 이하)에 비해 느린 수준으로 판단됩니다.<br/>배포 환경 최적화를 위해 GPU(CUDA) 환경으로 돌려보겠습니다.

### 5-2. GPU 진행을 위한 CUDA 12 + cuDNN 9 런타임 pip 설치 명령

In [ ]:
! pip install nvidia-cudnn-cu12 nvidia-cuda-runtime-cu12 nvidia-cublas-cu12

In [13]:
import os
from pathlib import Path

NVIDIA_BIN = Path(r'D:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\nvidia')
for sub in ['cudnn', 'cuda_runtime', 'cublas']:        # cuFFT/cuRAND는 DeepLab엔 불필요
    d = str(NVIDIA_BIN / sub / 'bin')
    if Path(d).is_dir():
        os.add_dll_directory(d)                                       # Python 3.8+ 공식
        os.environ['PATH'] = d + os.pathsep + os.environ['PATH']      # 전이 의존성용
        print(f'CUDA DLL dir added: {d}')
    else:
        print(f'(없음) {d}')

CUDA DLL dir added: D:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\nvidia\cudnn\bin
CUDA DLL dir added: D:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\nvidia\cuda_runtime\bin
CUDA DLL dir added: D:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\nvidia\cublas\bin


### 5-3. ORT CPU 벤치마크(GPU_CUDA)

In [15]:
# pip install onnxruntime-gpu  (기존 onnxruntime은 제거 권장: 동시 설치 시 충돌)
import onnxruntime as ort
print(ort.get_available_providers())  # 'CUDAExecutionProvider' 포함되어야 함

sess_gpu = ort.InferenceSession(
    str(ONNX_OUT),
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)

['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [17]:
import time
# warm-up + 측정
for _ in range(5):
    sess_gpu.run([output_name], {'input': x.numpy()})
t0 = time.perf_counter()
for _ in range(20):
    sess_gpu.run([output_name], {'input': x.numpy()})
print(f'ORT GPU 추론: {(time.perf_counter()-t0)/20*1000:.1f} ms/image')

ORT GPU 추론: 101.8 ms/image


## 6. 배포 환경 최적화 회고 (트러블슈팅 로그)

### 6-1. 결과 요약

| 환경 | latency | FPS | 비고 |
|---|---|---|---|
| ORT CPU FP32 | 532.2 ms | 1.9 | 베이스라인 (실시간 부적합) |
| **ORT GPU FP32 (CUDA EP)** | **102.1 ms** | **9.8** | **GPU 환경에서 실시간 처리 근접 수준 확보** |
| 가속비 | **5.2×** | — | 정확도 손실 0 (양자화 미적용) |

<br/>

> 초기 목표는 100 ms 이하 latency 확보였습니다.<br/> 측정 결과 평균 102 ms 수준으로 확인되었으며, 이는 warm-up 구간의 CUDA context 초기화 비용이 일부 포함된 결과로 판단하였습니다.<br/> 이후 IOBinding 적용 및 추가 최적화를 통해 추가 latency 감소 가능성이 있다고 판단하였습니다.

### 6-2. 의사결정 — 왜 GPU 경로를 먼저 택했는가

| 옵션 | 채택 여부 | 근거 |
|---|---|---|
| Dynamic Quantization | 미적용 | DeepLabV3+는 Conv 중심 구조이며 MatMul 비중이 거의 없어, ONNX Runtime Dynamic Quantization 적용 효과가 제한적일 것으로 판단하였습니다. 또한 defect recall 저하 가능성을 고려하였습니다. |
| Static Quantization (INT8) | 보류 | Calibration dataset 기반 정확도 검증이 추가로 필요하다고 판단하였습니다. CPU-only edge 환경 대응 옵션으로 유지하였습니다. |
| **ORT CUDA EP** | **채택** | 보유 GPU(1660 SUPER) 환경에서 별도 정확도 손실 없이 즉시 latency 개선 효과를 확인할 수 있어 우선 적용하였습니다. |

### 6-3. 시행착오 — 세 번의 막힘과 해결

| 단계 | 증상 | 원인 | 해결 |
|---|---|---|---|
| 1 | `get_available_providers()` 가 `[Azure, CPU]` 만 반환 | Jupyter 커널이 CPU-only onnxruntime 환경으로 연결되어 있었음을 확인하였습니다. | `venv_battery` 인터프리터 기반 커널로 변경 후 재시작하여 해결하였습니다. |
| 2 | CUDA provider 표시에도 latency 개선 없음 | ONNX Runtime이 cuDNN / CUDA runtime DLL을 찾지 못해 CPU provider로 fallback 되는 현상을 확인하였습니다. | `pip install nvidia-cudnn-cu12 nvidia-cuda-runtime-cu12 nvidia-cublas-cu12` (CUDA 12 / cuDNN 9 런타임 패키지를 추가 설치하였습니다.) |
| 3 | 패키지 설치 후에도 `Error 126: 모듈을 찾을 수 없음` | `os.add_dll_directory()` 만으로는 ORT CUDA EP 의 전이 의존성(cuDNN→cuBLAS) 해석을 보장하지 못하였습니다. | `os.add_dll_directory()` 와 `os.environ['PATH']` 방식을 적용하여 해결하였습니다. |

### 6-4. 핵심 정리

1. **`get_available_providers()` ≠ "GPU 작동 중"** — 실제 GPU 추론 여부를 판단할 수 없음을 확인하였습니다. 등록 가능한 provider와 실제 실행 `sess.get_providers()` 는 별도로 검증이 필요하였습니다.

2. **`onnxruntime-gpu` 설치 ≠ GPU 추론 가능** — ORT 1.20+ 는 cuDNN 9 / CUDA 12 런타임을 별도 요구합니다. NVIDIA
드라이버는 GPU 통신만 담당, 런타임 라이브러리는 별도 설치했습니다.

3. **ORT CUDA EP 는 실패해도 예외 없이 CPU 폴백** — 로그 한 줄만 남기고 silent fallback. 정확한 에러 추출을
위해 `providers=['CUDAExecutionProvider']` 단독으로 강제 호출하여, latency 측정과 provider 검증을 함께 수행해야 함을 확인하였습니다.

4. **Windows + Python 3.8+ DLL 로딩** — `os.add_dll_directory()` 는 Python이 직접 로드하는 DLL용. **전이
의존성 해석**은 PATH 도 같이 봐야 함. ORT CUDA EP 는 후자라 두 방식 모두 함께 필요함을 확인하였습니다.

5. **`-cu12` 접미사 ≠ 라이브러리 자체 버전 12** — `-cu12` 는 "CUDA 12용 빌드"라는 의미합니다. 실제 내부 라이브러리 버전과는 다를 수 있음을 확인하였습니다.

### 결론 및 다음 단계

사전 정의한 ONNX 변환 합격 기준(PyTorch ↔ ONNX max abs diff < 1e-3, 실제 결함 이미지 argmax 일치율 ≥ 99.9%,
GPU latency ≤ 100 ms)을 모두 충족하였으므로, 본 ONNX 모델(`models/battery_deeplab_v1.onnx` + `.onnx.data`)을 `Microsoft.ML.OnnxRuntime.Gpu` 기반 C# 데모에 통합하는 단계로 진행합니다.

배포 환경 GPU 활성화 과정의 트러블슈팅 상세는 `notebooks_docs/05_onnx_deployment_journey.md` 에 기록하였습니다.